## Лабораторный практикум по курсу «Разговорный ИИ и большие языковые модели», Университет ИТМО, 2026
**Лабораторная работа №5. Decoder-only: zero-shot и LoRA (Unsloth).**

**Модель:** [unsloth/Qwen3-4B-Instruct-2507](https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507).
Схема дообучения взята из [официального ноутбука Unsloth](https://github.com/unslothai/notebooks/blob/main/nb/Kaggle-Qwen3_(4B)-Instruct.ipynb): 4-bit, шаблон `qwen3-instruct`, LoRA, `train_on_responses_only`.
**Датасет:** [databricks/databricks-dolly-15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k). У него один сплит `train`, поэтому train / val / test режем сами.

**Задачи:**

1. Загрузить Qwen3-4B.
2. Разбить датасет на train / val / test.
3. Посчитать BLEU и METEOR в zero-shot на test.
4. Дообучить LoRA (val - контроль loss).
5. Посчитать те же метрики на том же test и сравнить.

Запуск проводился на Kaggle с T4.

In [1]:
# %%capture
!uv pip install pip3-autoremove
!uv pip install torch==2.12.1 torchvision==0.27.1 torchaudio==2.11.0 xformers==0.0.35 --index-url https://download.pytorch.org/whl/cu130
!uv pip install unsloth==2026.1.4 unsloth_zoo transformers==4.57.6 sacrebleu nltk pandas
!uv pip install --no-deps --upgrade "torchao>=0.16.0"
!uv pip install --no-deps trl==0.22.2

Using Python 3.13.16 environment at: /usr
Resolved 3 packages in 250ms                                         
Prepared 1 package in 24ms                                               
Installed 1 package in 2ms1.0                               
 + pip3-autoremove==2.1.0
Using Python 3.13.16 environment at: /usr
Resolved 34 packages in 553ms                                        
Prepared 7 packages in 17.22s                                            
Uninstalled 6 packages in 4.57s
Installed 7 packages in 274ms                               
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - torch==2.11.0+cu130
 + torch==2.12.1+cu130
 - torchvision==0.26.0+cu130
 + torchvision==0.27.1+cu130
 - triton==3.6.0
 + triton==3.7.1
 + xformers==0.0.35
Using Python 3.13.16 environment at: /usr
Resolved 99 packages in 2.19s                                 

In [ ]:
import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch
import pandas as pd
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from sacrebleu.metrics import BLEU
import nltk
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("omw-1.4", quiet=True)

In [ ]:
SEED = 42
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 256
# None = весь сплит. На бесплатном T4 полный test (~1.5k генераций) долгий.
N_TRAIN, N_VAL, N_TEST = 1000, 100, 50
GEN_BATCH = 4
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

cuda True Tesla T4


## Датасет: dolly-15k → train / val / test

In [8]:
raw = load_dataset("databricks/databricks-dolly-15k", split="train")
raw = raw.filter(lambda row: bool(row["instruction"].strip()) and bool(row["response"].strip()))
raw = raw.shuffle(seed=SEED)

n = len(raw)
n_test = n // 10
n_val = n // 10
test_raw = raw.select(range(0, n_test))
val_raw = raw.select(range(n_test, n_test + n_val))
train_raw = raw.select(range(n_test + n_val, n))

def take(ds, k):
    return ds if k is None else ds.select(range(min(k, len(ds))))

train_raw, val_raw, test_raw = take(train_raw, N_TRAIN), take(val_raw, N_VAL), take(test_raw, N_TEST)
print(f"full={n}  used train/val/test = {len(train_raw)}/{len(val_raw)}/{len(test_raw)}")
print(train_raw[0]["instruction"][:200])

full=15011  used train/val/test = 1000/100/50
Give me a list of nice board games to play with friends


## Модель

Загрузка квантованного 4-bit Qwen3-4B-Instruct. Шаблон чата — `qwen3-instruct`.

In [9]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")
tokenizer.padding_side = "left"

def user_text(row):
    text = row["instruction"].strip()
    ctx = (row["context"] or "").strip()
    return text if not ctx else text + "\n\n" + ctx

def to_conversations(examples):
    convos = []
    for instruction, context, response in zip(examples["instruction"], examples["context"], examples["response"]):
        user = instruction.strip()
        if context and context.strip():
            user = user + "\n\n" + context.strip()
        convos.append([
            {"role": "user", "content": user},
            {"role": "assistant", "content": response.strip()},
        ])
    return {"conversations": convos}

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["conversations"]
    ]
    return {"text": texts}

def with_text(ds):
    ds = ds.map(to_conversations, batched=True)
    return ds.map(formatting_prompts_func, batched=True)

train_ds = with_text(train_raw)
val_ds = with_text(val_raw)
print(train_ds[0]["text"][:800])

==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

<|im_start|>user
Give me a list of nice board games to play with friends<|im_end|>
<|im_start|>assistant
1. Cards Against Humanity
2. Codenames
3. The Game of Things
4. Taboo
5. Balderdash
6. Scattergories
7. Exploding Kittens<|im_end|>



## Генерация и метрики

`Dataset[i:j]` -- это словарь колонок, а не список строк. Батч берём через `select` и итерируем его.

In [10]:
def generate(rows):
    preds = []
    for start in range(0, len(rows), GEN_BATCH):
        batch = rows.select(range(start, min(start + GEN_BATCH, len(rows))))
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": user_text(row)}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for row in batch
        ]
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH,
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,  # для сопоставимого BLEU; для чата Unsloth советует temperature=0.7, top_p=0.8, top_k=20
                use_cache=True,
            )
        prompt_len = inputs["input_ids"].shape[1]
        for ids in out:
            preds.append(tokenizer.decode(ids[prompt_len:], skip_special_tokens=True).strip())
    return preds

def compute_bleu(preds, refs):
    return BLEU().corpus_score(preds, [refs]).score

def compute_meteor(preds, refs):
    scores = [meteor_score([ref.split()], pred.split()) for pred, ref in zip(preds, refs)]
    return float(sum(scores) / len(scores))

def evaluate(rows, title):
    preds = generate(rows)
    refs = [row["response"].strip() for row in rows]
    bleu, meteor = compute_bleu(preds, refs), compute_meteor(preds, refs)
    print(f"{title}: BLEU = {bleu:.2f}, METEOR = {meteor:.4f}")
    for i in range(min(3, len(rows))):
        print("-" * 60)
        print("Q:", user_text(rows[i])[:300])
        print("pred:", preds[i][:400])
        print("gold:", refs[i][:400])
    return {"bleu": bleu, "meteor": meteor}

## Zero-shot на test

In [11]:
zero_shot = evaluate(test_raw, "zero-shot")

/usr/local/lib/python3.13/dist-packages/unsloth/kernels/utils.py:963: UserWarning: An output with one or more elements was resized since it had shape [1, 2, 2560], which does not match the required output shape [2, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:31.)
  out = torch_matmul(X, W.t(), out = out)
/usr/local/lib/python3.13/dist-packages/unsloth/kernels/utils.py:970: UserWarning: An output with one or more elements was resized since it had shape [1, 2, 2560], which does not match the required output shape [2, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.res

zero-shot: BLEU = 3.52, METEOR = 0.2473
------------------------------------------------------------
Q: Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?
pred: Garth Greenhand is not a character in the *A Song of Ice and Fire* series by George R.R. Martin. There is no such figure as "Garth Greenhand" in the canon of the books or their associated lore.

It's possible there is a confusion with the name "Greenhand," which *is* a known name in the series. The most prominent figure with that name is **Greenhand**, a legendary warrior and the founder of the Gr
gold: Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree
------------------------------------------------------------
Q: Give me a list of basic ingr

## LoRA

Тюним вниманиеи слои проекции. loss только на ответе ассистента, маркеры чата библиотека берёт из шаблона сама. Обучение будет 1 эпоху.

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="no",
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        output_dir="lab5_lora",
    ),
)
trainer = train_on_responses_only(trainer)
trainer.train()
model.save_pretrained("qwen3-4b-lora")
tokenizer.save_pretrained("qwen3-4b-lora")

Unsloth 2026.1.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: reducing dataset_num_proc 8 -> 5 to fit free memory (~1GB per worker). Set UNSLOTH_DATASET_NUM_PROC to override.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/1000 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x7e3c845bce50>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 2215, in _audit_fork_safety
    def _audit_fork_safety(  # pragma: no cover - CPython disables tracing while Python audit hooks run
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7e3c2bbb1a30>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 2215, in _audit_fork_safety
    def _audit_fork_safety(  # pragma: no cover - CPython disables tracing while Python audit hooks run
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7e3c2bbb1cb0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 2215, in _audit_fork_safety
    def _audit_fork_safety(  # pragma: no cover - CPython disables tracing

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/100 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x7e3c62d3d0d0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 291, in __init__
    if hasattr(lock, '_release_save'):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7e3c84870130>
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 291, in __init__
    if hasattr(lock, '_release_save'):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7e3c2bbb22a0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 291, in __init__
    if hasattr(lock, '_release_save'):
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7e3c2bbb2930>
Traceback (most recent call last):
  File "/usr/lib/python3.13/threading.py", line 291, in __init__
    if hasattr(lock, '_release_save'):
Buffe

... (6 outputs left out)

Step,Training Loss,Validation Loss
50,2.362800,1.548311
100,1.750400,1.536113


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


('qwen3-4b-lora/tokenizer_config.json',
 'qwen3-4b-lora/special_tokens_map.json',
 'qwen3-4b-lora/chat_template.jinja',
 'qwen3-4b-lora/vocab.json',
 'qwen3-4b-lora/merges.txt',
 'qwen3-4b-lora/added_tokens.json',
 'qwen3-4b-lora/tokenizer.json')

## Test после LoRA и сравнение

In [13]:
finetuned = evaluate(test_raw, "LoRA")
table = pd.DataFrame({"zero-shot": zero_shot, "LoRA": finetuned}).T
table.loc["delta"] = table.loc["LoRA"] - table.loc["zero-shot"]
table.round(4)

/usr/local/lib/python3.13/dist-packages/unsloth/kernels/utils.py:963: UserWarning: An output with one or more elements was resized since it had shape [1, 4, 2560], which does not match the required output shape [4, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:31.)
  out = torch_matmul(X, W.t(), out = out)
/usr/local/lib/python3.13/dist-packages/unsloth/kernels/utils.py:970: UserWarning: An output with one or more elements was resized since it had shape [1, 4, 2560], which does not match the required output shape [4, 1, 2560]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.res

LoRA: BLEU = 7.80, METEOR = 0.2734
------------------------------------------------------------
Q: Who were the children of the legendary Garth Greenhand, the High King of the First Men in the series A Song of Ice and Fire?
pred: Garth Greenhand had two children, the first being the legendary Jon Snow, and the second being the infamous Arya Stark.
gold: Garth the Gardener, John the Oak, Gilbert of the Vines, Brandon of the Bloody Blade, Foss the Archer, Owen Oakenshield, Harlon the Hunter, Herndon of the Horn, Bors the Breaker, Florys the Fox, Maris the Maid, Rose of the Red Lake, Ellyn Ever Sweet, Rowan Gold-Tree
------------------------------------------------------------
Q: Give me a list of basic ingredients for baking cookies
pred: - Flour
- Butter
- Sugar
- Eggs
- Vanilla extract
- Baking soda
- Baking powder
- Salt
- Chocolate chips
gold: To bake cookies you will need flour, sugar, eggs, milk, butter, baking powder, and if you want; you can add chocolate chips, cinnamon, walnuts

,bleu,meteor
zero-shot,3.5240,0.2473
LoRA,7.7993,0.2734
delta,4.2753,0.0261


## Выводы

После всего одной эпохи QLoRA метрики стали лучше, ответы стали более похожими на изначальный датасет.

Из плюсов, конечно, экономия памяти, так как вместо условной квадратной матрицы 4096*4096 мы обучаем две матрицы меньшего размера 4096*r и r*4096, но в качестве trade off у нас скорость, поскольку на каждый блок LoRA у нас добавляется матричное умножение и добавление к замороженному выходу.